# Dagua UI Feature Playground

Interactive notebook for exercising the main user-facing setup and rendering features.

This notebook is meant for manual visual QA. It focuses on:
- graph construction paths
- layout and draw convenience APIs
- node, edge, and cluster styling
- edge routing and edge labels
- cluster labels and nested hierarchy
- flex constraints (pins and alignment)
- export paths

In [ ]:
from pathlib import Path

import torch
import dagua
from IPython.display import display
from dagua import DaguaGraph, LayoutConfig, NodeStyle, EdgeStyle, ClusterStyle
from dagua import Flex, LayoutFlex, AlignGroup

OUTDIR = Path('/tmp/dagua-ui-playground')
OUTDIR.mkdir(parents=True, exist_ok=True)
OUTDIR

## Graph Setup Helpers

These builders intentionally cover the main ways users initialize graphs.

In [ ]:
def make_manual_graph():
    g = DaguaGraph(direction='TB')
    g.add_node('input', label='Input', type='input')
    g.add_node('pre', label='Preprocess')
    g.add_node('branch_a', label='Branch A')
    g.add_node('branch_b', label='Branch B')
    g.add_node('merge', label='Merge')
    g.add_node('output', label='Output', type='output')
    g.add_edge('input', 'pre', label='clean + normalize')
    g.add_edge('pre', 'branch_a', label='left path')
    g.add_edge('pre', 'branch_b', label='right path')
    g.add_edge('branch_a', 'merge', label='contribution A')
    g.add_edge('branch_b', 'merge', label='contribution B')
    g.add_edge('merge', 'output', label='final output')
    g.node_styles[g._id_to_index['branch_a']] = NodeStyle(shape='ellipse')
    g.node_styles[g._id_to_index['branch_b']] = NodeStyle(shape='diamond')
    g.edge_styles[1] = EdgeStyle(routing='ortho', port_style='center')
    g.edge_styles[2] = EdgeStyle(routing='straight', curvature=0.0)
    g.add_cluster('Preparation Stage', ['pre', 'branch_a', 'branch_b'], label='Preparation Stage')
    g.compute_node_sizes()
    return g


def make_from_edge_list_graph():
    g = DaguaGraph.from_edge_list([
        ('x', 'TokenEmbedding(vocab=50257, dim=4096)'),
        ('TokenEmbedding(vocab=50257, dim=4096)', 'LayerNorm(normalized_shape=(4096,))'),
        ('LayerNorm(normalized_shape=(4096,))', 'Q'),
        ('LayerNorm(normalized_shape=(4096,))', 'K'),
        ('LayerNorm(normalized_shape=(4096,))', 'V'),
        ('Q', 'Attention'),
        ('K', 'Attention'),
        ('V', 'Attention'),
        ('Attention', '+'),
        ('x', '+'),
        ('+', 'MLP(hidden_dim=16384)'),
        ('MLP(hidden_dim=16384)', 'out'),
    ])
    g.edge_labels = [
        'embed', 'normalize', 'query path', 'key path', 'value path',
        'attend', 'attend', 'attend', 'residual merge', 'shortcut', 'expand', 'emit',
    ]
    g.compute_node_sizes()
    return g


def make_from_edge_index_graph():
    edge_index = torch.tensor([
        [0, 0, 0, 1, 2, 3, 4, 5],
        [1, 2, 3, 4, 4, 5, 6, 6],
    ], dtype=torch.long)
    g = DaguaGraph.from_edge_index(edge_index, num_nodes=7)
    g.node_labels = ['hub', 'tiny', 'medium', 'long', 'merge', 'tail', 'sink']
    g.node_styles[0] = NodeStyle(shape='ellipse')
    g.node_styles[3] = NodeStyle(shape='diamond')
    g.edge_labels = [
        'a', 'b', 'c', 'tiny merge', 'medium merge', 'late path', 'to sink', 'final',
    ]
    g.compute_node_sizes()
    return g


def make_nested_cluster_graph():
    g = DaguaGraph.from_edge_list([
        ('input', 'encoder.stage0'),
        ('encoder.stage0', 'encoder.stage1.attn'),
        ('encoder.stage1.attn', 'encoder.stage1.mlp'),
        ('encoder.stage1.mlp', 'bridge'),
        ('bridge', 'decoder.cross_attn'),
        ('decoder.cross_attn', 'decoder.ffn'),
        ('decoder.ffn', 'output'),
    ])
    idx = {name: i for i, name in enumerate(g.node_labels)}
    g.add_cluster(
        'Encoder Cluster With Long Title',
        [idx['encoder.stage0'], idx['encoder.stage1.attn'], idx['encoder.stage1.mlp']],
        label='Encoder Cluster With Long Title',
        style=ClusterStyle(label_offset=(10.0, 16.0)),
    )
    g.add_cluster(
        'Encoder Inner Block',
        [idx['encoder.stage1.attn'], idx['encoder.stage1.mlp']],
        label='Encoder Inner Block',
        parent='Encoder Cluster With Long Title',
        style=ClusterStyle(label_offset=(10.0, 14.0)),
    )
    g.add_cluster(
        'Decoder Cluster With Another Long Title',
        [idx['decoder.cross_attn'], idx['decoder.ffn']],
        label='Decoder Cluster With Another Long Title',
        style=ClusterStyle(label_offset=(10.0, 16.0)),
    )
    g.add_edge('encoder.stage1.attn', 'bridge', label='attention handoff')
    g.add_edge('bridge', 'decoder.cross_attn', label='decoder context transfer')
    g.compute_node_sizes()
    return g


def make_flex_graph():
    g = DaguaGraph.from_edge_list([
        ('input', 'a'), ('input', 'b'), ('input', 'c'),
        ('a', 'join'), ('b', 'join'), ('c', 'join'),
        ('join', 'output'),
    ])
    g.compute_node_sizes()
    g.flex = LayoutFlex(
        pins={
            'input': (Flex.locked(0.0), Flex.locked(0.0)),
            'output': (Flex.locked(240.0), Flex.locked(180.0)),
        },
        align_x=[AlignGroup(['a', 'b', 'c'], weight=5.0)],
    )
    return g


GRAPH_BUILDERS = {
    'manual_graph': make_manual_graph,
    'from_edge_list': make_from_edge_list_graph,
    'from_edge_index': make_from_edge_index_graph,
    'nested_clusters': make_nested_cluster_graph,
    'flex_constraints': make_flex_graph,
}

list(GRAPH_BUILDERS)

## Pick A Graph And Draw It

Change `graph_name`, `direction`, `theme_name`, `device`, or `steps` and rerun.

In [ ]:
graph_name = 'manual_graph'
direction = 'TB'
theme_name = 'default'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
steps = 80
edge_opt_steps = -1

dagua.reset()
dagua.set_theme(theme_name)

g = GRAPH_BUILDERS[graph_name]()
g.direction = direction
config = LayoutConfig(device=device, steps=steps, direction=direction, edge_opt_steps=edge_opt_steps, seed=42)
fig, ax = dagua.draw(g, config)
fig

## Inspect The Lower-Level Pipeline

This uses the same steps as `dagua.draw()` but exposes intermediate objects for debugging.

In [ ]:
g = GRAPH_BUILDERS['nested_clusters']()
config = LayoutConfig(device=device, steps=60, direction='TB', edge_opt_steps=-1, seed=42)
positions = dagua.layout(g, config)
curves = dagua.route_edges(positions, g.edge_index, g.node_sizes, g.direction, g)
label_positions = dagua.place_edge_labels(curves, positions, g.node_sizes, g.edge_labels, g)
fig, ax = dagua.render(g, positions, config, curves=curves, label_positions=label_positions)
fig

## Routing And Label Stress

This is useful for edge label collisions, routing mode tweaks, and shape-aware ports.

In [ ]:
g = make_manual_graph()
g.edge_styles[0] = EdgeStyle(routing='straight', port_style='center', label_position=0.25)
g.edge_styles[1] = EdgeStyle(routing='ortho', port_style='distributed', label_position=0.6)
g.edge_styles[2] = EdgeStyle(routing='bezier', curvature=0.8, label_position=0.4)
g.edge_styles[3] = EdgeStyle(routing='bezier', curvature=0.2, label_position=0.55)
config = LayoutConfig(device=device, steps=70, direction='TB', edge_opt_steps=-1, seed=42)
fig, ax = dagua.draw(g, config)
fig

## Flex Constraints Demo

Shows hard pins and within-layer alignment.

In [ ]:
g = GRAPH_BUILDERS['flex_constraints']()
config = LayoutConfig(device=device, steps=90, direction='TB', edge_opt_steps=-1, seed=42)
fig, ax = dagua.draw(g, config)
fig

## Theme And Direction Sweep

Quick way to eyeball the same graph under different user-visible presentation settings.

In [ ]:
sweep = [
    ('default', 'TB'),
    ('minimal', 'LR'),
    ('dark', 'TB'),
]

for theme_name, direction in sweep:
    dagua.reset()
    dagua.set_theme(theme_name)
    g = GRAPH_BUILDERS['from_edge_list']()
    g.direction = direction
    config = LayoutConfig(device=device, steps=60, direction=direction, edge_opt_steps=-1, seed=42)
    fig, ax = dagua.draw(g, config)
    fig.suptitle(f'{theme_name} / {direction}')
    display(fig)


## Export Checks

Useful for quick manual verification that save/export paths still work.

In [ ]:
g = GRAPH_BUILDERS['manual_graph']()
json_path = OUTDIR / 'manual_graph.json'
yaml_path = OUTDIR / 'manual_graph.yaml'
png_path = OUTDIR / 'manual_graph.png'

dagua.save(g, json_path)
dagua.save(g, yaml_path)
dagua.draw(g, LayoutConfig(device=device, steps=60, edge_opt_steps=-1, seed=42), output=png_path)

json_path, yaml_path, png_path